# Resolución: cuántas muestras y cuántos niveles

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura oficial:** Procesamiento de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Medir cómo se degrada una imagen cuando se reduce su resolución espacial —cuántos píxeles tiene— y su resolución de intensidad —cuántos niveles distintos puede tomar cada píxel—, y aprender a distinguir las dos pérdidas.

## Al terminar este cuaderno van a poder:

1. Submuestrear una imagen con slicing de paso y calcular cuántos píxeles conserva.
2. Explicar la diferencia entre submuestrear y achicar con `cv2.resize` e `INTER_AREA`.
3. Cuantizar una imagen a una cantidad elegida de niveles.
4. Contar los niveles presentes con `np.unique` y construir un histograma con `cv2.calcHist`.
5. Reconocer las bandas de falsos contornos como síntoma de una cuantización insuficiente.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Muestreo espacial** | Cuántos píxeles se usan para representar la escena: más píxeles, más detalle posible. |
| **Paso de muestreo** | Cada cuántos píxeles nos quedamos con uno. Paso 4 significa uno de cada cuatro, en filas y en columnas. |
| **Cuantización** | Cuántos valores distintos de intensidad puede tomar cada píxel. |
| **Nivel de intensidad** | Cada uno de los valores posibles de un píxel en grises: en `uint8` hay 256, del 0 al 255. |
| **Histograma** | Un conteo de cuántos píxeles tienen cada nivel de intensidad. |
| **Falso contorno** | Una banda o escalón visible en un degradado suave, que aparece cuando faltan niveles intermedios. |

## Dos formas de perder información

**Analogía.** Una imagen digital es como un mosaico. Se puede empobrecer de dos maneras: usando baldosas más grandes, con lo que caben menos y se pierde el dibujo fino, o usando menos colores de baldosa, con lo que los degradados se vuelven escalones. La primera es un problema de **muestreo**; la segunda, de **cuantización**.

**Bajada tecnológica.** Los megapíxeles de una cámara describen su muestreo. La cantidad de bits por píxel describe su cuantización: una foto común usa 8 bits por canal, mientras que las imágenes médicas y científicas suelen usar 12 o 16 bits para distinguir diferencias sutiles de intensidad.

### ⚙ Configuración del entorno

In [ ]:
from pathlib import Path

import cv2
import numpy as np

from utils import mostrar_imágenes, mostrar_curvas

print(f"OpenCV {cv2.__version__} — NumPy {np.__version__}")

En este cuaderno trabajamos en escala de grises, para concentrarnos en la intensidad. Definimos una función de lectura en grises, siguiendo el mismo esquema que `leer_imagen_rgb`.

In [ ]:
CARPETA_IMÁGENES = Path("..") / "imagenes"

def leer_imagen_gris(ruta_de_la_imagen):
    """Lee una imagen en escala de grises con OpenCV y verifica la lectura."""
    imagen_leída = cv2.imread(filename=str(ruta_de_la_imagen), flags=cv2.IMREAD_GRAYSCALE)
    assert imagen_leída is not None, f"No se pudo leer: {ruta_de_la_imagen}"
    return imagen_leída

cartas_gris = leer_imagen_gris(CARPETA_IMÁGENES / "cards.jpg")
globos_gris = leer_imagen_gris(CARPETA_IMÁGENES / "globos.jpg")

forma_cartas = cartas_gris.shape
forma_globos = globos_gris.shape
print(f"Cartas: {forma_cartas} | Globos: {forma_globos}")

Elegimos dos imágenes con propósitos distintos. Las cartas tienen detalle fino —letras, figuras, bordes de dibujos—, ideal para ver qué se pierde al submuestrear. Los globos tienen un cielo con un degradado suave, ideal para ver qué se pierde al cuantizar.

## 1. Muestreo: quedarse con menos píxeles

**Analogía.** Una encuesta que consulta a una de cada cuatro personas de un barrio capta bien la tendencia general, pero puede perderse lo que piensa un grupo chico. Submuestrear una imagen es consultar a uno de cada cuatro píxeles.

**Bajada tecnológica.** Las miniaturas de una galería de fotos y las vistas previas de un video son versiones submuestreadas. En imágenes satelitales, la resolución espacial se expresa en metros por píxel: con 10 metros por píxel se ve un campo, pero no un auto.

El slicing con paso, `imagen[::paso, ::paso]`, se queda con un píxel cada `paso` filas y cada `paso` columnas.

### ✎ Antes de ejecutar

Las cartas tienen 500 × 477 píxeles. Con paso 4, ¿qué proporción de los píxeles originales va a quedar: la mitad, un cuarto o un dieciseisavo?

In [ ]:
pasos_a_probar = [1, 2, 4, 8]
cartas_submuestreadas = []

for paso in pasos_a_probar:
    # '::paso' toma un elemento cada 'paso' posiciones, en filas y en columnas.
    cartas_con_paso = cartas_gris[::paso, ::paso]
    cantidad_de_píxeles = cartas_con_paso.size
    forma_con_paso = cartas_con_paso.shape
    print(f"Paso {paso}: forma {forma_con_paso} → {cantidad_de_píxeles:,} píxeles")
    cartas_submuestreadas.append({"imagen": cartas_con_paso, "título": f"Paso {paso}: {forma_con_paso}"})

Cada vez que duplicamos el paso, la cantidad de píxeles se divide por cuatro, porque se reducen a la vez las filas y las columnas. Con paso 4 queda un dieciseisavo; con paso 8, apenas uno de cada 64 píxeles.

Mostramos las cuatro versiones con el mismo tamaño en pantalla. Así la versión con menos píxeles se ve estirada y deja en evidencia qué detalle perdió.

In [ ]:
mostrar_imágenes(cartas_submuestreadas)

Con paso 2 las cartas se siguen leyendo sin problema. Con paso 4 las letras y los palos todavía se reconocen, pero los dibujos finos de las figuras empiezan a verse como manchas. Con paso 8 la imagen se vuelve una grilla de cuadrados: se distinguen las cuatro cartas, pero no su contenido.

### Traducción: esto se conecta con `004_encuadre`

El slicing con paso descarta píxeles, igual que `cv2.resize` con `INTER_NEAREST`. En el cuaderno anterior vimos que `INTER_AREA`, en cambio, promedia. Comparamos las dos maneras de llegar al mismo tamaño.

In [ ]:
cartas_paso_8 = cartas_gris[::8, ::8]
alto_reducido = cartas_paso_8.shape[0]
ancho_reducido = cartas_paso_8.shape[1]

# 'INTER_AREA' calcula cada píxel nuevo como el promedio del bloque que reemplaza.
cartas_promediadas = cv2.resize(src=cartas_gris, dsize=(ancho_reducido, alto_reducido), interpolation=cv2.INTER_AREA)

paneles = [
    {"imagen": cartas_paso_8, "título": "Slicing con paso 8: descarta"},
    {"imagen": cartas_promediadas, "título": "INTER_AREA: promedia"},
]
mostrar_imágenes(paneles, alto_de_figura=5)

Las dos imágenes tienen exactamente la misma cantidad de píxeles, pero no se ven igual. La versión con slicing muestra bordes cortados y puntos sueltos, porque cada píxel es una muestra aislada. La versión promediada es más suave y representa mejor las zonas claras y oscuras. **La cantidad de muestras es la misma; lo que cambia es cómo se eligió cada muestra.**

### ✎ Para pensar

- Si submuestrean una imagen con paso 8 y después la agrandan otra vez a su tamaño original, ¿recuperan el detalle de las figuras? ¿Por qué?
- ¿Qué pasa con una línea diagonal fina de un solo píxel de ancho cuando se submuestrea con paso 2?

## 2. Cuantización: quedarse con menos niveles

**Analogía.** Con una caja de 256 lápices de grises se puede pintar un cielo con un degradado casi perfecto. Con una caja de 4 lápices se puede pintar el mismo cielo, pero el degradado se convierte en franjas. El dibujo sigue siendo reconocible; lo que se perdió son los tonos intermedios.

**Bajada tecnológica.** El formato GIF limita cada imagen a 256 colores, y por eso las fotos en GIF muestran bandas. Las pantallas de tinta electrónica de los lectores de libros usan pocos niveles de gris. Y los modelos de inteligencia artificial se "cuantizan" para ocupar menos memoria, con la misma idea aplicada a otros números.

Para cuantizar a una cantidad de niveles, dividimos el rango de 0 a 255 en escalones iguales y reemplazamos cada píxel por el centro de su escalón. Lo escribimos primero paso a paso, con 4 niveles.

In [ ]:
cantidad_de_niveles = 4
tamaño_del_escalón = 256 // cantidad_de_niveles

# La división entera '//' dice en qué escalón cae cada píxel: 0, 1, 2 o 3.
número_de_escalón = globos_gris // tamaño_del_escalón

# Cada píxel toma el valor del centro de su escalón.
inicio_del_escalón = número_de_escalón * tamaño_del_escalón
globos_4_niveles = inicio_del_escalón + tamaño_del_escalón // 2

niveles_presentes = np.unique(ar=globos_4_niveles)
print(f"Tamaño de cada escalón: {tamaño_del_escalón}")
print(f"Niveles presentes en la imagen cuantizada: {niveles_presentes}")

Cada escalón abarca 64 valores. Todos los píxeles entre 0 y 63 pasan a valer 32, los que están entre 64 y 127 pasan a 96, y así. `np.unique` confirma que la imagen quedó con exactamente cuatro valores distintos.

Ahora que el procedimiento se entiende, lo encapsulamos para poder probar varias cantidades de niveles.

In [ ]:
def cuantizar(imagen_gris, cantidad_de_niveles):
    """Reduce una imagen en grises a una cantidad de niveles igualmente espaciados.

    Cada píxel se reemplaza por el valor central del escalón en el que cae.
    """
    tamaño_del_escalón = 256 // cantidad_de_niveles
    número_de_escalón = imagen_gris // tamaño_del_escalón
    inicio_del_escalón = número_de_escalón * tamaño_del_escalón
    imagen_cuantizada = inicio_del_escalón + tamaño_del_escalón // 2
    return imagen_cuantizada

print("Función cuantizar definida.")

### ✎ Antes de ejecutar

Vamos a cuantizar los globos a 256, 16, 8 y 4 niveles. ¿En qué zona de la imagen esperan que aparezcan primero las bandas: en los globos o en el cielo?

In [ ]:
niveles_a_probar = [256, 16, 8, 4]
globos_cuantizados = []

for niveles in niveles_a_probar:
    globos_con_niveles = cuantizar(imagen_gris=globos_gris, cantidad_de_niveles=niveles)
    valores_distintos = np.unique(ar=globos_con_niveles)
    cantidad_de_valores = valores_distintos.size
    print(f"Pedimos {niveles} niveles → la imagen tiene {cantidad_de_valores} valores distintos")
    globos_cuantizados.append({"imagen": globos_con_niveles, "título": f"{niveles} niveles"})

mostrar_imágenes(globos_cuantizados)

Con 16 niveles los globos todavía se ven naturales, pero el cielo —que en el original es un degradado continuo— ya muestra franjas diagonales. Con 8 niveles las franjas se ensanchan y los globos empiezan a tener zonas planas. Con 4 niveles el cielo queda en dos o tres bandas y los globos parecen recortes de papel. Esas franjas son **falsos contornos**: parecen bordes, pero no existen en la escena.

La cuenta de valores distintos puede dar menos que los niveles pedidos: si la imagen original no usa todo el rango de 0 a 255, algunos escalones quedan vacíos.

El histograma permite ver la cuantización en números. `cv2.calcHist` cuenta cuántos píxeles hay en cada nivel de intensidad.

In [ ]:
def calcular_histograma(imagen_gris):
    """Cuenta cuántos píxeles tiene cada uno de los 256 niveles de intensidad."""
    # 'channels=[0]' usa el único canal; 'histSize=[256]' crea un casillero por nivel.
    conteo = cv2.calcHist(images=[imagen_gris], channels=[0], mask=None, histSize=[256], ranges=[0, 256])
    # 'calcHist' devuelve una columna de 256 filas; 'ravel' la aplana en una lista de 256 valores.
    conteo_plano = conteo.ravel()
    return conteo_plano

globos_8_niveles = cuantizar(imagen_gris=globos_gris, cantidad_de_niveles=8)

histograma_original = calcular_histograma(imagen_gris=globos_gris)
histograma_8_niveles = calcular_histograma(imagen_gris=globos_8_niveles)

total_de_píxeles = histograma_original.sum()
print(f"Histogramas calculados. Cada uno suma {total_de_píxeles:,.0f} píxeles.")

Los dos histogramas suman la misma cantidad de píxeles: cuantizar no quita píxeles, solo los reubica en menos niveles. Los graficamos juntos para comparar su forma.

In [ ]:
curvas = [
    {"valores": histograma_original, "nombre": "Original (256 niveles)"},
    {"valores": histograma_8_niveles, "nombre": "Cuantizada a 8 niveles"},
]
mostrar_curvas(curvas=curvas, título="Histograma de los globos en grises", etiqueta_x="Nivel de intensidad", etiqueta_y="Cantidad de píxeles")

El histograma original es una curva continua y baja, con una joroba ancha entre 130 y 180 que corresponde sobre todo al cielo, la zona más grande de la imagen. Se ve baja porque sus píxeles están repartidos entre muchos niveles. El histograma cuantizado se reduce a ocho picos aislados separados por huecos vacíos: todos los píxeles de cada escalón se apilaron sobre un único valor, y por eso cada pico es mucho más alto. Los huecos son justamente los tonos intermedios que desaparecieron.

### ✎ Modificación guiada

En la celda donde se cuantizan los globos, cambien la lista `niveles_a_probar` por `[6, 3, 2]` y vuelvan a ejecutarla. Antes, anoten con cuántos niveles creen que los globos dejan de distinguirse del cielo.

### ✎ Para pensar

- ¿Por qué el cielo muestra falsos contornos antes que los globos, si los dos se cuantizaron con los mismos escalones?
- Una imagen en `uint8` tiene 256 niveles, es decir, 8 bits por píxel. ¿Cuántos bits harían falta para guardar una imagen de 16 niveles? ¿Y una de 2?

### ✎ Completado

Comprueben la relación entre niveles y bits con los datos. Completen la celda para calcular cuántos bits necesita cada cantidad de niveles de la lista.

In [ ]:
# ─── Completen los pasos faltantes ───────────────────────────────────────────
#
# TODO 1: recorran la lista 'niveles_a_probar' con un for.
# TODO 2: para cada cantidad de niveles, calculen los bits necesarios con np.log2.
# TODO 3: muestren con print la cantidad de niveles y la cantidad de bits.
#
# for niveles in niveles_a_probar:
#     bits_necesarios = ...
#     print(...)
#
# ─────────────────────────────────────────────────────────────────────────────

## 3. Dos pérdidas distintas, lado a lado

Para cerrar, comparamos las dos pérdidas sobre la misma imagen de las cartas. Elegimos dos reducciones que ahorran información de maneras diferentes: quedarnos con un píxel de cada 16 (paso 4) y quedarnos con 4 niveles de 256.

In [ ]:
cartas_menos_píxeles = cartas_gris[::4, ::4]
cartas_menos_niveles = cuantizar(imagen_gris=cartas_gris, cantidad_de_niveles=4)

paneles = [
    {"imagen": cartas_gris, "título": "Original"},
    {"imagen": cartas_menos_píxeles, "título": "Menos muestras: paso 4"},
    {"imagen": cartas_menos_niveles, "título": "Menos niveles: 4 niveles"},
]
mostrar_imágenes(paneles)

Las dos versiones perdieron información, pero de naturaleza distinta. La submuestreada conserva todos los tonos, pero pierde **dónde** están los detalles finos: bordes, letras, adornos. La cuantizada conserva la posición de los detalles finos de las figuras, pero pierde **cuánta** luz hay en cada zona: los grises intermedios se funden en pocos tonos planos, y la textura de la madera se convierte en manchas negras.

### ✎ Para pensar

- Si tuvieran que leer el valor de cada carta desde lejos, ¿cuál de las dos pérdidas les resultaría más tolerable? ¿Y si tuvieran que distinguir la textura de la madera de la mesa?

## Actividad de producción

In [ ]:
# ─── Espacio de práctica ─────────────────────────────────────────────────────
#
# Consigna: encuentren el límite de utilidad de una imagen.
#   1. Lean small-boats_1.jpg en escala de grises.
#   2. Busquen el paso de muestreo más grande con el que todavía se puede
#      contar a las dos personas del bote.
#   3. Busquen la menor cantidad de niveles con la que todavía se distinguen
#      los nenúfares del agua.
#   4. Muestren las dos versiones límite y escriban en una celda de texto
#      qué criterio usaron para decidir que "todavía se distingue".
#
# ─────────────────────────────────────────────────────────────────────────────

## Cierre

Una imagen puede perder información en el espacio o en la intensidad. Las dos pérdidas se miden de forma distinta y producen defectos distintos.

| | Muestreo | Cuantización |
|---|---|---|
| Qué se reduce | La cantidad de píxeles | La cantidad de niveles por píxel |
| Cómo se hace en este cuaderno | Slicing con paso o `cv2.resize` | División entera y escalones |
| Qué defecto aparece | Pérdida de detalle fino, bordes dentados | Falsos contornos, tonos planos |
| Cómo se detecta | Mirando la forma y los detalles | Con `np.unique` y con el histograma |

**Limitaciones de lo que vimos.** Usamos cuantización uniforme, con escalones del mismo tamaño en todo el rango. Existen métodos que eligen los niveles según la imagen y conservan mejor los tonos más frecuentes. Además, aunque la imagen cuantizada tenga pocos niveles, en memoria sigue ocupando un byte por píxel: la reducción es de información, no de espacio en disco, salvo que el formato de archivo la aproveche.

En el próximo cuaderno, `006_anotacion`, pasamos de medir la imagen a escribir sobre ella: rectángulos, círculos, líneas y texto para dejar marcados los resultados.